# Geopolitical Risk and the Real Economy: Industry-Level Evidence
## From Employment, Prices, and Output During the 2026 Strait of Hormuz Crisis

**Paper Objective:** Examine how geopolitical risk (GPR) shocks transmit to the real US economy at the industry level, using the February 28, 2026 Iran conflict as a natural experiment.

**Data Sources:**
- FRED (Federal Reserve Economic Data) — Macro indicators, GPR index, oil prices
- BLS (Bureau of Labor Statistics) — Employment, CPI, PPI by industry
- BEA (Bureau of Economic Analysis) — GDP by industry
- Alpha Vantage — Sector ETF daily prices
- Treasury Fiscal Data — Government spending, revenue, debt
- FMP (Financial Modeling Prep) — Sector financial ratios

**Methodology:**
1. Event Study — Sector ETF abnormal returns around Feb 28, 2026
2. Panel VAR — Industry-level impulse responses to GPR shocks
3. Granger Causality — Financial markets → real economy transmission
4. Cross-sectional Regression — What sector characteristics explain heterogeneous responses?

---

## Section 1: Setup and Configuration

In [ ]:
# ============================================================
# 1.1  Install dependencies (run once in Colab)
# ============================================================
!pip install pandas numpy requests matplotlib seaborn scipy statsmodels scikit-learn arch -q

In [ ]:
# ============================================================
# 1.2  Imports and configuration
# ============================================================
import pandas as pd
import numpy as np
import requests
import time
import warnings
import json
from datetime import datetime, timedelta
from io import StringIO

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})
sns.set_style("whitegrid")

print("All imports successful.")

In [ ]:
# ============================================================
# 1.3  API Keys  —  ENTER YOUR KEYS HERE
# ============================================================
API_KEYS = {
    'FRED':           '5f33ed2b539cfd80cd4a5343654a6fca',
    'ALPHA_VANTAGE':  'ZV1GXYVC2K8627SE',
    'BLS':            '5df8024e059b4966a66b69f1224259de',
    'BEA':            '94A4A4DE-3C3B-42F8-9204-6118E0A244DF',
    'FMP':            'J2cyfrrQiwQf1rhS06wzRSPTU8Pq2ybz',
}

TREASURY_BASE = 'https://api.fiscaldata.treasury.gov/services/api/fiscal_service'

# Key dates for the analysis
EVENT_DATE = pd.Timestamp('2026-02-28')          # US-Israel strikes on Iran
SAMPLE_START = '2023-01-01'                       # Pre-crisis baseline start
SAMPLE_END = '2026-03-03'                         # Current date

print(f"Event date:    {EVENT_DATE.date()}")
print(f"Sample window: {SAMPLE_START} to {SAMPLE_END}")

In [ ]:
# ============================================================
# 1.4  Helper functions for API calls with retry logic
# ============================================================
def api_get(url, params=None, retries=3, delay=2):
    """GET request with exponential backoff retry."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp
        except requests.exceptions.RequestException as e:
            if attempt < retries - 1:
                wait = delay * (2 ** attempt)
                print(f"  Retry {attempt+1}/{retries} after {wait}s ... ({e})")
                time.sleep(wait)
            else:
                print(f"  FAILED after {retries} attempts: {e}")
                return None
    return None

def safe_float(val):
    """Convert value to float, returning NaN on failure."""
    try:
        return float(val)
    except (ValueError, TypeError):
        return np.nan

print("Helper functions ready.")

---
## Section 2: Data Collection

We collect data from **six sources**. Each cell is self-contained — if one API is down, others still work.

### 2.1 FRED — Macro Indicators and Geopolitical Risk Index

We collect:
- **GPRH**: Geopolitical Risk Index (daily) — Caldara & Iacoviello
- **DCOILWTICO**: WTI Crude Oil Price (daily)
- **DCOILBRENTEU**: Brent Crude Oil Price (daily)
- **DFF**: Federal Funds Effective Rate (daily)
- **DGS10**: 10-Year Treasury Yield (daily)
- **VIXCLS**: CBOE Volatility Index (daily)
- **DTWEXBGS**: Trade-Weighted US Dollar Index (daily)
- **UMCSENT**: University of Michigan Consumer Sentiment (monthly)
- **UNRATE**: Unemployment Rate (monthly)
- **CPIAUCSL**: Consumer Price Index (monthly)
- **INDPRO**: Industrial Production Index (monthly)

In [ ]:
# ============================================================
# 2.1  FRED Data Collection
# ============================================================
def fetch_fred_series(series_id, start=SAMPLE_START, end=SAMPLE_END):
    """Fetch a single FRED time series."""
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series_id,
        'api_key': API_KEYS['FRED'],
        'file_type': 'json',
        'observation_start': start,
        'observation_end': end,
    }
    resp = api_get(url, params=params)
    if resp is None:
        print(f"  [WARN] Could not fetch FRED series: {series_id}")
        return pd.Series(dtype=float, name=series_id)

    data = resp.json()
    obs = data.get('observations', [])
    if not obs:
        print(f"  [WARN] No observations for FRED series: {series_id}")
        return pd.Series(dtype=float, name=series_id)

    df = pd.DataFrame(obs)
    df['date'] = pd.to_datetime(df['date'])
    df['value'] = df['value'].apply(safe_float)
    df = df.set_index('date')['value'].dropna()
    df.name = series_id
    return df

# Define all FRED series we need
FRED_DAILY_SERIES = {
    'GPRH':          'Geopolitical Risk Index (Daily)',
    'DCOILWTICO':    'WTI Crude Oil ($/barrel)',
    'DCOILBRENTEU':  'Brent Crude Oil ($/barrel)',
    'DFF':           'Fed Funds Rate (%)',
    'DGS10':         '10-Year Treasury Yield (%)',
    'VIXCLS':        'VIX Index',
    'DTWEXBGS':      'USD Trade-Weighted Index',
}

FRED_MONTHLY_SERIES = {
    'UMCSENT':   'Consumer Sentiment (Michigan)',
    'UNRATE':    'Unemployment Rate (%)',
    'CPIAUCSL':  'CPI All Urban Consumers',
    'INDPRO':    'Industrial Production Index',
}

# Fetch daily series
print("Fetching FRED daily series...")
fred_daily = {}
for sid, label in FRED_DAILY_SERIES.items():
    print(f"  {sid}: {label}")
    fred_daily[sid] = fetch_fred_series(sid)
    time.sleep(0.3)  # Rate limiting

fred_daily_df = pd.DataFrame(fred_daily)
print(f"\nFRED daily: {fred_daily_df.shape[0]} rows, {fred_daily_df.shape[1]} columns")
print(f"Date range: {fred_daily_df.index.min().date()} to {fred_daily_df.index.max().date()}")

# Fetch monthly series
print("\nFetching FRED monthly series...")
fred_monthly = {}
for sid, label in FRED_MONTHLY_SERIES.items():
    print(f"  {sid}: {label}")
    fred_monthly[sid] = fetch_fred_series(sid)
    time.sleep(0.3)

fred_monthly_df = pd.DataFrame(fred_monthly)
print(f"\nFRED monthly: {fred_monthly_df.shape[0]} rows, {fred_monthly_df.shape[1]} columns")
print(f"Date range: {fred_monthly_df.index.min().date()} to {fred_monthly_df.index.max().date()}")

# Quick peek
print("\n--- FRED Daily (last 5 rows) ---")
display(fred_daily_df.tail())
print("\n--- FRED Monthly (last 5 rows) ---")
display(fred_monthly_df.tail())

### 2.2 Alpha Vantage — Sector ETF Daily Prices

The 11 Select Sector SPDR ETFs map to GICS sectors. We also collect SPY (S&P 500) as the market benchmark.

In [ ]:
# ============================================================
# 2.2  Alpha Vantage — Sector ETFs (Daily)
# ============================================================
SECTOR_ETFS = {
    'SPY':  'S&P 500 (Market)',
    'XLE':  'Energy',
    'XLF':  'Financials',
    'XLI':  'Industrials',
    'XLK':  'Technology',
    'XLV':  'Health Care',
    'XLC':  'Communication Services',
    'XLB':  'Materials',
    'XLRE': 'Real Estate',
    'XLU':  'Utilities',
    'XLP':  'Consumer Staples',
    'XLY':  'Consumer Discretionary',
}

def fetch_alpha_vantage_daily(symbol):
    """Fetch daily adjusted close prices from Alpha Vantage."""
    url = 'https://www.alphavantage.co/query'
    params = {
        'function': 'TIME_SERIES_DAILY',
        'symbol': symbol,
        'outputsize': 'full',
        'apikey': API_KEYS['ALPHA_VANTAGE'],
    }
    resp = api_get(url, params=params)
    if resp is None:
        return pd.Series(dtype=float, name=symbol)

    data = resp.json()

    # Check for rate limit message
    if 'Note' in data or 'Information' in data:
        msg = data.get('Note', data.get('Information', ''))
        print(f"  [RATE LIMIT] {symbol}: {msg[:80]}...")
        return pd.Series(dtype=float, name=symbol)

    ts_key = 'Time Series (Daily)'
    if ts_key not in data:
        print(f"  [WARN] No daily data for {symbol}. Keys: {list(data.keys())}")
        return pd.Series(dtype=float, name=symbol)

    ts = data[ts_key]
    series = pd.Series(
        {pd.Timestamp(dt): safe_float(vals['4. close']) for dt, vals in ts.items()},
        name=symbol
    ).sort_index()

    # Filter to our sample window
    series = series.loc[SAMPLE_START:SAMPLE_END]
    return series

print("Fetching Alpha Vantage sector ETF data...")
print("(Alpha Vantage free tier: 25 calls/day — may need multiple days for full data)\n")

etf_data = {}
for symbol, sector_name in SECTOR_ETFS.items():
    print(f"  {symbol}: {sector_name}", end=" ... ")
    series = fetch_alpha_vantage_daily(symbol)
    if len(series) > 0:
        etf_data[symbol] = series
        print(f"{len(series)} days")
    else:
        print("NO DATA")
    time.sleep(13)  # Alpha Vantage free tier: 5 calls/min

etf_df = pd.DataFrame(etf_data)
etf_df.index = pd.to_datetime(etf_df.index)
etf_df = etf_df.sort_index()

print(f"\nETF data: {etf_df.shape[0]} trading days, {etf_df.shape[1]} ETFs")
if not etf_df.empty:
    print(f"Date range: {etf_df.index.min().date()} to {etf_df.index.max().date()}")
    display(etf_df.tail())

### 2.3 BLS — Employment, CPI Components, and PPI by Industry

Using BLS Public Data API v2 (registered key = 500 queries/day, 50 series per query).

In [ ]:
# ============================================================
# 2.3  BLS Data Collection
# ============================================================
def fetch_bls_series(series_ids, start_year=2023, end_year=2026):
    """
    Fetch multiple BLS series via API v2 (POST, registered key).
    Max 50 series per request.
    """
    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    headers = {'Content-type': 'application/json'}
    payload = json.dumps({
        'seriesid': series_ids,
        'startyear': str(start_year),
        'endyear': str(end_year),
        'registrationkey': API_KEYS['BLS'],
    })

    resp = requests.post(url, data=payload, headers=headers, timeout=30)
    if resp.status_code != 200:
        print(f"  [ERROR] BLS API returned {resp.status_code}")
        return {}

    result = resp.json()
    if result.get('status') != 'REQUEST_SUCCEEDED':
        print(f"  [ERROR] BLS: {result.get('message', 'Unknown error')}")
        return {}

    all_series = {}
    for series_data in result.get('Results', {}).get('series', []):
        sid = series_data['seriesID']
        records = []
        for obs in series_data.get('data', []):
            year = int(obs['year'])
            period = obs['period']
            if not period.startswith('M'):
                continue
            month = int(period[1:])
            date = pd.Timestamp(year=year, month=month, day=1)
            val = safe_float(obs['value'])
            records.append({'date': date, 'value': val})

        if records:
            s = pd.DataFrame(records).set_index('date')['value'].sort_index()
            s.name = sid
            all_series[sid] = s

    return all_series

# ------------------------------------------------------------------
# A) CES Employment by major supersector (Seasonally adjusted,
#    All employees, thousands)
#    Series ID format: CES{supersector}01
# ------------------------------------------------------------------
CES_EMPLOYMENT = {
    'CES0500000001': 'Total Private',
    'CES1000000001': 'Mining & Logging',
    'CES2000000001': 'Construction',
    'CES3000000001': 'Manufacturing',
    'CES4000000001': 'Trade, Transport, Utilities',
    'CES4142000001': 'Retail Trade',
    'CES5000000001': 'Information',
    'CES5500000001': 'Financial Activities',
    'CES6000000001': 'Professional & Business Svcs',
    'CES6500000001': 'Education & Health Svcs',
    'CES7000000001': 'Leisure & Hospitality',
    'CES8000000001': 'Other Services',
    'CES9000000001': 'Government',
}

# B) CPI components (SA, 1982-84=100)
CPI_COMPONENTS = {
    'CUSR0000SA0':    'CPI All Items',
    'CUSR0000SA0E':   'CPI Energy',
    'CUSR0000SAF1':   'CPI Food',
    'CUSR0000SAH1':   'CPI Shelter',
    'CUSR0000SAM':    'CPI Medical',
    'CUSR0000SAT1':   'CPI Transportation',
    'CUSR0000SAA':    'CPI Apparel',
}

# C) PPI by industry (selected, NSA)
PPI_INDUSTRY = {
    'PCU2111--2111--':   'PPI Oil & Gas Extraction',
    'PCU325---325---':   'PPI Chemicals',
    'PCU333---333---':   'PPI Machinery',
    'PCU336---336---':   'PPI Transportation Equip',
    'PCU221---221---':   'PPI Utilities',
    'PCU484---484---':   'PPI Truck Transportation',
    'PCU523---523---':   'PPI Securities & Investing',
}

print("Fetching BLS Employment (CES) data...")
ces_series = fetch_bls_series(list(CES_EMPLOYMENT.keys()))
ces_df = pd.DataFrame(ces_series)
ces_df.columns = [CES_EMPLOYMENT.get(c, c) for c in ces_df.columns]
print(f"  Employment: {ces_df.shape}")

time.sleep(1)

print("Fetching BLS CPI component data...")
cpi_series = fetch_bls_series(list(CPI_COMPONENTS.keys()))
cpi_df = pd.DataFrame(cpi_series)
cpi_df.columns = [CPI_COMPONENTS.get(c, c) for c in cpi_df.columns]
print(f"  CPI: {cpi_df.shape}")

time.sleep(1)

print("Fetching BLS PPI industry data...")
ppi_series = fetch_bls_series(list(PPI_INDUSTRY.keys()))
ppi_df = pd.DataFrame(ppi_series)
ppi_df.columns = [PPI_INDUSTRY.get(c, c) for c in ppi_df.columns]
print(f"  PPI: {ppi_df.shape}")

# Display summaries
print("\n--- Employment (last 5 months) ---")
display(ces_df.tail())
print("\n--- CPI Components (last 5 months) ---")
display(cpi_df.tail())
print("\n--- PPI Industries (last 5 months) ---")
display(ppi_df.tail())

### 2.4 BEA — GDP by Industry

In [ ]:
# ============================================================
# 2.4  BEA — GDP by Industry (Quarterly)
# ============================================================
def fetch_bea_gdp_by_industry():
    """
    Fetch GDP by industry from BEA API.
    Dataset: GDPByIndustry, TableID 1 (Value Added by Industry)
    """
    url = 'https://apps.bea.gov/api/data/'
    params = {
        'UserID': API_KEYS['BEA'],
        'method': 'GetData',
        'DatasetName': 'GDPByIndustry',
        'TableID': '1',
        'Frequency': 'Q',
        'Year': ','.join([str(y) for y in range(2023, 2027)]),
        'Industry': 'ALL',
        'ResultFormat': 'JSON',
    }
    resp = api_get(url, params=params)
    if resp is None:
        print("  [WARN] BEA API request failed")
        return pd.DataFrame()

    data = resp.json()

    # Navigate BEA's nested JSON structure
    results = data.get('BEAAPI', {}).get('Results', {})
    if isinstance(results, list):
        results = results[0] if results else {}

    rows = results.get('Data', [])
    if not rows:
        # Try alternate path
        rows = data.get('BEAAPI', {}).get('Results', [{}])
        if isinstance(rows, dict):
            rows = rows.get('Data', [])
        print(f"  [INFO] BEA returned {len(rows)} rows")

    if not rows:
        print("  [WARN] No BEA data returned")
        return pd.DataFrame()

    records = []
    for row in rows:
        try:
            year = int(row.get('Year', 0))
            quarter_str = row.get('Quarter', '')
            industry = row.get('IndustrYDescription', row.get('IndustryDescription', ''))
            value = safe_float(row.get('DataValue', '').replace(',', ''))

            # Parse quarter to date
            q_map = {'I': 1, 'II': 4, 'III': 7, 'IV': 10,
                      '1': 1, '2': 4, '3': 7, '4': 10}
            month = q_map.get(quarter_str.replace('Q', ''), None)
            if month is None:
                continue
            date = pd.Timestamp(year=year, month=month, day=1)

            records.append({
                'date': date,
                'industry': industry,
                'gdp_value_added': value
            })
        except (ValueError, KeyError):
            continue

    if not records:
        print("  [WARN] Could not parse any BEA records")
        return pd.DataFrame()

    df = pd.DataFrame(records)
    return df

print("Fetching BEA GDP by Industry data...")
bea_raw = fetch_bea_gdp_by_industry()
print(f"  BEA raw: {bea_raw.shape}")

if not bea_raw.empty:
    # Pivot to wide format: rows=date, columns=industry
    bea_df = bea_raw.pivot_table(
        index='date', columns='industry', values='gdp_value_added', aggfunc='first'
    )
    bea_df = bea_df.sort_index()

    # Show the industries we got
    print(f"  Industries: {bea_df.shape[1]}")
    print(f"  Date range: {bea_df.index.min().date()} to {bea_df.index.max().date()}")
    print(f"\n  Industry list:")
    for i, col in enumerate(bea_df.columns[:20], 1):
        print(f"    {i}. {col}")
    if bea_df.shape[1] > 20:
        print(f"    ... and {bea_df.shape[1]-20} more")
else:
    bea_df = pd.DataFrame()
    print("  [WARN] No BEA data available — will proceed with other sources")

### 2.5 Treasury Fiscal Data — Federal Spending, Revenue, Debt

In [ ]:
# ============================================================
# 2.5  Treasury Fiscal Data
# ============================================================
def fetch_treasury_data(endpoint, fields, date_field='record_date',
                         start=SAMPLE_START, end=SAMPLE_END, page_size=1000):
    """Fetch data from Treasury Fiscal Data API."""
    url = f"{TREASURY_BASE}/{endpoint}"
    params = {
        'fields': ','.join([date_field] + fields),
        'filter': f'{date_field}:gte:{start},{date_field}:lte:{end}',
        'sort': f'-{date_field}',
        'page[number]': 1,
        'page[size]': page_size,
    }
    resp = api_get(url, params=params)
    if resp is None:
        return pd.DataFrame()

    data = resp.json().get('data', [])
    if not data:
        return pd.DataFrame()

    df = pd.DataFrame(data)
    df[date_field] = pd.to_datetime(df[date_field])
    df = df.set_index(date_field).sort_index()

    for col in fields:
        if col in df.columns:
            df[col] = df[col].apply(safe_float)

    return df

# A) Monthly Treasury Statement — Outlays & Receipts
print("Fetching Treasury Monthly Statement (Outlays & Receipts)...")
treasury_mts = fetch_treasury_data(
    endpoint='v1/accounting/mts/mts_table_5',
    fields=['current_month_gross_outly_amt', 'current_month_rcpt_amt',
            'current_fytd_gross_outly_amt', 'current_fytd_rcpt_amt',
            'classification_desc']
)
print(f"  MTS: {treasury_mts.shape}")

# B) Debt to the Penny (daily outstanding federal debt)
print("Fetching Treasury Debt to the Penny...")
treasury_debt = fetch_treasury_data(
    endpoint='v2/accounting/od/debt_to_penny',
    fields=['tot_pub_debt_out_amt']
)
print(f"  Debt: {treasury_debt.shape}")

if not treasury_debt.empty:
    print(f"  Latest debt: ${treasury_debt['tot_pub_debt_out_amt'].iloc[-1]/1e12:.2f} trillion")
    display(treasury_debt.tail())
else:
    print("  [INFO] Treasury debt data may be sparse — API returns recent data only")

### 2.6 FMP — Sector Financial Ratios and Earnings

Financial Modeling Prep gives us firm-level data. We use sector-level aggregates. **(250 calls/day limit — use sparingly)**

In [ ]:
# ============================================================
# 2.6  FMP — Sector Performance and Key Ratios
# ============================================================
FMP_BASE = 'https://financialmodelingprep.com/api/v3'

def fetch_fmp_sector_performance():
    """Fetch sector performance (daily changes) from FMP."""
    url = f"{FMP_BASE}/sector-performance"
    params = {'apikey': API_KEYS['FMP']}
    resp = api_get(url, params=params)
    if resp is None:
        return pd.DataFrame()
    data = resp.json()
    if isinstance(data, list):
        return pd.DataFrame(data)
    return pd.DataFrame()

def fetch_fmp_sector_pe():
    """Fetch sector P/E ratios from FMP."""
    url = f"{FMP_BASE}/sector_price_earning_ratio"
    params = {
        'date': EVENT_DATE.strftime('%Y-%m-%d'),
        'exchange': 'NYSE',
        'apikey': API_KEYS['FMP'],
    }
    resp = api_get(url, params=params)
    if resp is None:
        return pd.DataFrame()
    data = resp.json()
    if isinstance(data, list):
        return pd.DataFrame(data)
    return pd.DataFrame()

def fetch_fmp_historical_sector():
    """Fetch historical sector performance from FMP."""
    url = f"{FMP_BASE}/historical-sectors-performance"
    params = {
        'limit': 200,
        'apikey': API_KEYS['FMP'],
    }
    resp = api_get(url, params=params)
    if resp is None:
        return pd.DataFrame()
    data = resp.json()
    if isinstance(data, list):
        return pd.DataFrame(data)
    return pd.DataFrame()

# Fetch current sector performance
print("Fetching FMP sector data...")
fmp_sector_perf = fetch_fmp_sector_performance()
print(f"  Sector performance: {fmp_sector_perf.shape}")
if not fmp_sector_perf.empty:
    display(fmp_sector_perf)

time.sleep(1)

# Fetch sector P/E ratios
fmp_sector_pe = fetch_fmp_sector_pe()
print(f"  Sector P/E ratios: {fmp_sector_pe.shape}")
if not fmp_sector_pe.empty:
    display(fmp_sector_pe.head(15))

time.sleep(1)

# Fetch historical sector performance
fmp_hist = fetch_fmp_historical_sector()
print(f"  Historical sector performance: {fmp_hist.shape}")
if not fmp_hist.empty:
    fmp_hist['date'] = pd.to_datetime(fmp_hist['date'])
    fmp_hist = fmp_hist.set_index('date').sort_index()
    # Convert percentage strings to floats
    for col in fmp_hist.columns:
        fmp_hist[col] = fmp_hist[col].apply(
            lambda x: safe_float(str(x).replace('%', '')) if pd.notna(x) else np.nan
        )
    display(fmp_hist.tail())

---
## Section 3: Data Processing and Construction of Key Variables

We construct:
1. **Daily returns** for sector ETFs (log returns)
2. **Abnormal returns** relative to market model
3. **Monthly percentage changes** for employment, CPI, PPI
4. **GPR shock indicator** — dummy variable for post-Feb 28, 2026
5. **Merged panel dataset** for econometric analysis

In [ ]:
# ============================================================
# 3.1  Compute Daily Log Returns for Sector ETFs
# ============================================================
if not etf_df.empty:
    etf_returns = np.log(etf_df / etf_df.shift(1)).dropna()
    print(f"ETF daily returns: {etf_returns.shape}")
    print(f"Date range: {etf_returns.index.min().date()} to {etf_returns.index.max().date()}")

    # Summary statistics
    print("\n--- Return Summary Statistics ---")
    summary = etf_returns.describe().T
    summary['skewness'] = etf_returns.skew()
    summary['kurtosis'] = etf_returns.kurtosis()
    display(summary[['mean', 'std', 'min', 'max', 'skewness', 'kurtosis']].round(4))
else:
    etf_returns = pd.DataFrame()
    print("[WARN] No ETF data — will use FMP historical sector data as fallback")

In [ ]:
# ============================================================
# 3.2  Compute Monthly Percentage Changes for Real Economy Data
# ============================================================

# Employment month-over-month growth (%)
if not ces_df.empty:
    emp_growth = ces_df.pct_change() * 100
    emp_growth.columns = [f"{c} (%MoM)" for c in emp_growth.columns]
    print(f"Employment growth: {emp_growth.shape}")
else:
    emp_growth = pd.DataFrame()

# CPI month-over-month inflation (%)
if not cpi_df.empty:
    cpi_inflation = cpi_df.pct_change() * 100
    cpi_inflation.columns = [f"{c} (%MoM)" for c in cpi_inflation.columns]
    print(f"CPI inflation: {cpi_inflation.shape}")
else:
    cpi_inflation = pd.DataFrame()

# PPI month-over-month change (%)
if not ppi_df.empty:
    ppi_change = ppi_df.pct_change() * 100
    ppi_change.columns = [f"{c} (%MoM)" for c in ppi_change.columns]
    print(f"PPI change: {ppi_change.shape}")
else:
    ppi_change = pd.DataFrame()

# Show latest data
if not emp_growth.empty:
    print("\n--- Employment Growth (last 3 months) ---")
    display(emp_growth.dropna().tail(3))

In [ ]:
# ============================================================
# 3.3  Construct GPR Shock Variables
# ============================================================

# Daily GPR change
if 'GPRH' in fred_daily_df.columns:
    fred_daily_df['GPR_change'] = fred_daily_df['GPRH'].pct_change() * 100
    fred_daily_df['GPR_log'] = np.log(fred_daily_df['GPRH'].replace(0, np.nan))

# Oil price returns
if 'DCOILWTICO' in fred_daily_df.columns:
    fred_daily_df['WTI_return'] = np.log(fred_daily_df['DCOILWTICO'] / fred_daily_df['DCOILWTICO'].shift(1)) * 100

# Post-event dummy
fred_daily_df['post_event'] = (fred_daily_df.index >= EVENT_DATE).astype(int)

# Monthly GPR (average of daily)
if 'GPRH' in fred_daily_df.columns:
    gpr_monthly = fred_daily_df['GPRH'].resample('MS').mean()
    gpr_monthly.name = 'GPR_monthly'
    gpr_monthly_change = gpr_monthly.pct_change() * 100
    gpr_monthly_change.name = 'GPR_monthly_change'
    print(f"Monthly GPR: {len(gpr_monthly)} observations")
    print(f"\nGPR around event date:")
    event_window = fred_daily_df.loc['2026-02-20':'2026-03-03', ['GPRH', 'GPR_change', 'post_event']]
    if not event_window.empty:
        display(event_window)
    else:
        print("  (Event window data not yet available — will appear after FRED updates)")
else:
    gpr_monthly = pd.Series(dtype=float)
    gpr_monthly_change = pd.Series(dtype=float)
    print("[INFO] GPRH not available from FRED — using oil price as proxy for geopolitical shock")

In [ ]:
# ============================================================
# 3.4  Build Merged Monthly Panel Dataset
# ============================================================

# Combine all monthly data into one panel
monthly_frames = []

# FRED monthly
if not fred_monthly_df.empty:
    monthly_frames.append(fred_monthly_df)

# GPR monthly average
if len(gpr_monthly) > 0:
    monthly_frames.append(gpr_monthly.to_frame())
    monthly_frames.append(gpr_monthly_change.to_frame())

# Monthly oil price (average of daily)
if 'DCOILWTICO' in fred_daily_df.columns:
    oil_monthly = fred_daily_df['DCOILWTICO'].resample('MS').mean()
    oil_monthly.name = 'WTI_monthly'
    oil_monthly_ret = oil_monthly.pct_change() * 100
    oil_monthly_ret.name = 'WTI_monthly_return'
    monthly_frames.append(oil_monthly.to_frame())
    monthly_frames.append(oil_monthly_ret.to_frame())

# Monthly VIX average
if 'VIXCLS' in fred_daily_df.columns:
    vix_monthly = fred_daily_df['VIXCLS'].resample('MS').mean()
    vix_monthly.name = 'VIX_monthly'
    monthly_frames.append(vix_monthly.to_frame())

# Merge all
if monthly_frames:
    monthly_macro = pd.concat(monthly_frames, axis=1)
    monthly_macro = monthly_macro.sort_index()
    monthly_macro['post_event'] = (monthly_macro.index >= EVENT_DATE).astype(int)
    print(f"Monthly macro panel: {monthly_macro.shape}")
    print(f"Columns: {list(monthly_macro.columns)}")
    display(monthly_macro.tail())
else:
    monthly_macro = pd.DataFrame()
    print("[WARN] No monthly data assembled")

---
## Section 4: Event Study — Sector Abnormal Returns Around Feb 28, 2026

**Methodology**: Market model event study (MacKinlay, 1997)
- **Estimation window**: [-250, -11] trading days (roughly 1 year before the event)
- **Event window**: [-10, +10] trading days around Feb 28, 2026
- For each sector ETF, estimate: $R_{i,t} = \alpha_i + \beta_i R_{m,t} + \epsilon_{i,t}$
- **Abnormal Return**: $AR_{i,t} = R_{i,t} - (\hat{\alpha}_i + \hat{\beta}_i R_{m,t})$
- **Cumulative Abnormal Return**: $CAR_i[t_1, t_2] = \sum_{t=t_1}^{t_2} AR_{i,t}$
- Statistical significance via cross-sectional t-test

In [ ]:
# ============================================================
# 4.1  Event Study: Market Model Estimation and Abnormal Returns
# ============================================================

def run_event_study(returns_df, market_col='SPY', event_date=EVENT_DATE,
                    est_window=(-250, -11), evt_window=(-10, 10)):
    """
    Run a market model event study.

    Parameters:
    -----------
    returns_df : DataFrame of daily log returns (sectors + market)
    market_col : column name for market benchmark
    event_date : the event date (Timestamp)
    est_window : tuple (start, end) relative trading days for estimation
    evt_window : tuple (start, end) relative trading days for event window

    Returns:
    --------
    results : dict with AR, CAR, statistics per sector
    """
    if returns_df.empty or market_col not in returns_df.columns:
        print("[ERROR] Cannot run event study — missing return data")
        return None

    # Find event date index in trading calendar
    trading_days = returns_df.index.sort_values()
    event_idx = trading_days.searchsorted(event_date)

    # If exact date not found, use nearest trading day
    if event_idx >= len(trading_days):
        event_idx = len(trading_days) - 1
    if trading_days[event_idx] != event_date:
        # Find closest trading day on or after event
        mask = trading_days >= event_date
        if mask.any():
            event_idx = mask.argmax()
        else:
            event_idx = len(trading_days) - 1
    actual_event_date = trading_days[event_idx]
    print(f"Event date (nearest trading day): {actual_event_date.date()}")

    # Define windows
    est_start = max(0, event_idx + est_window[0])
    est_end = max(0, event_idx + est_window[1])
    evt_start = max(0, event_idx + evt_window[0])
    evt_end = min(len(trading_days) - 1, event_idx + evt_window[1])

    est_dates = trading_days[est_start:est_end + 1]
    evt_dates = trading_days[evt_start:evt_end + 1]

    print(f"Estimation window: {est_dates[0].date()} to {est_dates[-1].date()} ({len(est_dates)} days)")
    print(f"Event window:      {evt_dates[0].date()} to {evt_dates[-1].date()} ({len(evt_dates)} days)")

    sector_cols = [c for c in returns_df.columns if c != market_col]
    market_ret = returns_df[market_col]

    results = {}
    for sector in sector_cols:
        sector_ret = returns_df[sector]

        # Estimation window data
        est_data = pd.DataFrame({
            'market': market_ret.loc[est_dates],
            'sector': sector_ret.loc[est_dates]
        }).dropna()

        if len(est_data) < 30:
            print(f"  [SKIP] {sector}: insufficient estimation data ({len(est_data)} obs)")
            continue

        # OLS: R_sector = alpha + beta * R_market
        X_est = sm.add_constant(est_data['market'])
        model = sm.OLS(est_data['sector'], X_est).fit()
        alpha_hat = model.params.iloc[0]
        beta_hat = model.params.iloc[1]
        sigma_est = model.resid.std()

        # Event window abnormal returns
        evt_data = pd.DataFrame({
            'market': market_ret.loc[evt_dates],
            'sector': sector_ret.loc[evt_dates],
        }).dropna()

        if evt_data.empty:
            print(f"  [SKIP] {sector}: no event window data")
            continue

        evt_data['expected'] = alpha_hat + beta_hat * evt_data['market']
        evt_data['AR'] = evt_data['sector'] - evt_data['expected']
        evt_data['CAR'] = evt_data['AR'].cumsum()

        # Standardized AR (for significance testing)
        evt_data['SAR'] = evt_data['AR'] / sigma_est

        # t-statistics for CAR
        n_evt = len(evt_data)
        car_value = evt_data['AR'].sum()
        car_std = sigma_est * np.sqrt(n_evt)
        t_stat = car_value / car_std if car_std > 0 else 0
        p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(est_data) - 2))

        # Relative day index
        evt_data['rel_day'] = range(-len(evt_data[evt_data.index < actual_event_date]),
                                     len(evt_data) - len(evt_data[evt_data.index < actual_event_date]))

        results[sector] = {
            'alpha': alpha_hat,
            'beta': beta_hat,
            'sigma': sigma_est,
            'r_squared': model.rsquared,
            'AR': evt_data[['AR', 'CAR', 'SAR']].copy(),
            'CAR_total': car_value,
            'CAR_t_stat': t_stat,
            'CAR_p_value': p_value,
            'evt_data': evt_data,
        }
        sig = "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.10 else ""
        print(f"  {sector:5s}: beta={beta_hat:6.3f}, CAR={car_value*100:+7.2f}%, "
              f"t={t_stat:+6.3f}, p={p_value:.4f} {sig}")

    return results

# Run the event study
print("="*70)
print("EVENT STUDY: Sector Abnormal Returns Around Feb 28, 2026")
print("="*70)

if not etf_returns.empty and 'SPY' in etf_returns.columns:
    event_results = run_event_study(etf_returns)
else:
    event_results = None
    print("\n[INFO] ETF data not available. Event study will be run after")
    print("       Alpha Vantage data is collected (may take multiple Colab sessions")
    print("       due to 25 calls/day free tier limit).")

In [ ]:
# ============================================================
# 4.2  Event Study — Publication-Quality Table
# ============================================================

if event_results:
    # Build summary table
    summary_rows = []
    for sector, res in event_results.items():
        sector_name = SECTOR_ETFS.get(sector, sector)
        sig = ""
        if res['CAR_p_value'] < 0.01: sig = "***"
        elif res['CAR_p_value'] < 0.05: sig = "**"
        elif res['CAR_p_value'] < 0.10: sig = "*"

        summary_rows.append({
            'Sector': sector_name,
            'Ticker': sector,
            'Beta': round(res['beta'], 3),
            'R-squared': round(res['r_squared'], 3),
            'CAR [-10,+10] (%)': round(res['CAR_total'] * 100, 2),
            't-statistic': round(res['CAR_t_stat'], 3),
            'p-value': round(res['CAR_p_value'], 4),
            'Significance': sig,
        })

    event_table = pd.DataFrame(summary_rows)
    event_table = event_table.sort_values('CAR [-10,+10] (%)', ascending=True)

    print("="*70)
    print("TABLE 1: Cumulative Abnormal Returns by Sector — Event Window [-10,+10]")
    print("Event: US-Israel Military Strikes on Iran (February 28, 2026)")
    print("="*70)
    print("Significance: *** p<0.01, ** p<0.05, * p<0.10")
    print()
    display(event_table.set_index('Sector'))
else:
    print("[INFO] Event study results not available yet — run Section 2.2 first")

In [ ]:
# ============================================================
# 4.3  Event Study — CAR Plot (Figure 1 for the paper)
# ============================================================

if event_results:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Panel A: Individual sector CARs
    ax1 = axes[0]
    colors = plt.cm.tab10(np.linspace(0, 1, len(event_results)))
    for (sector, res), color in zip(event_results.items(), colors):
        evt = res['evt_data']
        label = f"{SECTOR_ETFS.get(sector, sector)}"
        ax1.plot(evt['rel_day'], evt['CAR'] * 100, label=label, color=color, linewidth=1.5)

    ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Event Day')
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax1.set_xlabel('Trading Days Relative to Event')
    ax1.set_ylabel('Cumulative Abnormal Return (%)')
    ax1.set_title('Panel A: Sector CARs Around Feb 28, 2026')
    ax1.legend(fontsize=8, loc='lower left')
    ax1.grid(True, alpha=0.3)

    # Panel B: Bar chart of total CARs with significance
    ax2 = axes[1]
    sectors_sorted = sorted(event_results.items(), key=lambda x: x[1]['CAR_total'])
    names = [SECTOR_ETFS.get(s, s) for s, _ in sectors_sorted]
    cars = [r['CAR_total'] * 100 for _, r in sectors_sorted]
    p_vals = [r['CAR_p_value'] for _, r in sectors_sorted]

    bar_colors = ['#d32f2f' if c < 0 else '#388e3c' for c in cars]
    edge_colors = ['black' if p < 0.05 else 'gray' for p in p_vals]
    linewidths = [2 if p < 0.05 else 0.5 for p in p_vals]

    bars = ax2.barh(names, cars, color=bar_colors, edgecolor=edge_colors, linewidth=linewidths)
    ax2.axvline(x=0, color='black', linewidth=0.8)
    ax2.set_xlabel('Cumulative Abnormal Return (%)')
    ax2.set_title('Panel B: Total CAR [-10,+10] by Sector')

    # Add significance stars
    for i, (name, car, p_val) in enumerate(zip(names, cars, p_vals)):
        sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.10 else ""
        offset = 0.3 if car >= 0 else -0.3
        ax2.text(car + offset, i, sig, va='center', fontsize=10, fontweight='bold')

    ax2.grid(True, axis='x', alpha=0.3)

    plt.suptitle('Figure 1: Event Study — Sector Abnormal Returns Around the 2026 Iran Conflict',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('figure1_event_study.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Figure saved: figure1_event_study.png")
else:
    print("[INFO] Skipping plot — no event study results")

---
## Section 5: VAR Analysis and Granger Causality

**Objective**: How does a geopolitical risk shock transmit to the real economy over time?

We estimate a reduced-form VAR model using monthly data:
$$Y_t = c + A_1 Y_{t-1} + A_2 Y_{t-2} + \ldots + A_p Y_{t-p} + u_t$$

Where $Y_t$ = [GPR, Oil Price, VIX, Industrial Production, Unemployment, CPI]

We then compute:
1. **Impulse Response Functions (IRFs)** — how each variable responds to a GPR shock over 12 months
2. **Forecast Error Variance Decomposition (FEVD)** — what share of variation in each variable is explained by GPR shocks
3. **Granger Causality** — does GPR predict real economy variables?

In [ ]:
# ============================================================
# 5.1  Prepare Monthly VAR Dataset
# ============================================================

def check_stationarity(series, name, significance=0.05):
    """Run ADF test and report results."""
    clean = series.dropna()
    if len(clean) < 10:
        return name, np.nan, np.nan, 'Insufficient data'
    result = adfuller(clean, autolag='AIC')
    adf_stat, p_value = result[0], result[1]
    status = 'Stationary' if p_value < significance else 'Non-stationary'
    return name, adf_stat, p_value, status

# Build VAR dataset from monthly macro data
var_candidates = {}

# GPR
if len(gpr_monthly) > 0:
    var_candidates['GPR'] = gpr_monthly_change  # Use % change for stationarity
elif 'WTI_monthly_return' in monthly_macro.columns:
    print("[INFO] Using WTI oil return as GPR proxy")

# Oil
if 'WTI_monthly_return' in monthly_macro.columns:
    var_candidates['Oil_Return'] = monthly_macro['WTI_monthly_return']

# VIX
if 'VIX_monthly' in monthly_macro.columns:
    var_candidates['VIX'] = monthly_macro['VIX_monthly']

# Industrial Production (% change)
if 'INDPRO' in monthly_macro.columns:
    var_candidates['IndProd_Growth'] = monthly_macro['INDPRO'].pct_change() * 100

# Unemployment rate (already in levels, typically I(1) so use change)
if 'UNRATE' in monthly_macro.columns:
    var_candidates['Unemp_Change'] = monthly_macro['UNRATE'].diff()

# CPI inflation
if 'CPIAUCSL' in monthly_macro.columns:
    var_candidates['CPI_Inflation'] = monthly_macro['CPIAUCSL'].pct_change() * 100

# Consumer Sentiment change
if 'UMCSENT' in monthly_macro.columns:
    var_candidates['Sentiment_Change'] = monthly_macro['UMCSENT'].pct_change() * 100

# Build clean VAR dataframe
var_df = pd.DataFrame(var_candidates).dropna()
print(f"VAR dataset: {var_df.shape[0]} months, {var_df.shape[1]} variables")
print(f"Variables: {list(var_df.columns)}")
print(f"Date range: {var_df.index.min().date()} to {var_df.index.max().date()}")

# Stationarity tests
print("\n--- ADF Stationarity Tests ---")
print(f"{'Variable':<20s} {'ADF Stat':>10s} {'p-value':>10s} {'Status':<20s}")
print("-" * 60)
for col in var_df.columns:
    name, adf, pval, status = check_stationarity(var_df[col], col)
    pval_str = f"{pval:.4f}" if not np.isnan(pval) else "N/A"
    adf_str = f"{adf:.4f}" if not np.isnan(adf) else "N/A"
    print(f"{name:<20s} {adf_str:>10s} {pval_str:>10s} {status:<20s}")

display(var_df.tail())

In [ ]:
# ============================================================
# 5.2  Estimate VAR Model
# ============================================================

if var_df.shape[0] >= 20 and var_df.shape[1] >= 3:
    # Select optimal lag length using information criteria
    max_lags = min(6, var_df.shape[0] // 5)  # Rule of thumb
    var_model_select = VAR(var_df)

    print("--- Lag Order Selection ---")
    lag_results = var_model_select.select_order(maxlags=max_lags)
    print(lag_results.summary())

    # Use AIC-selected lag (with minimum of 1)
    optimal_lag = max(1, lag_results.aic)
    print(f"\nSelected lag order (AIC): {optimal_lag}")

    # Estimate VAR
    var_model = var_model_select.fit(optimal_lag)
    print(f"\nVAR({optimal_lag}) estimated with {var_model.nobs} observations")
    print(f"Variables: {list(var_model.names)}")

    # Model summary
    print("\n--- VAR Model Summary ---")
    print(var_model.summary())
else:
    var_model = None
    print(f"[WARN] Insufficient data for VAR: {var_df.shape[0]} obs, {var_df.shape[1]} vars")
    print("       Need at least 20 monthly observations and 3 variables")

In [ ]:
# ============================================================
# 5.3  Impulse Response Functions (Figure 2 for the paper)
# ============================================================

if var_model is not None:
    # Compute IRFs — shock to GPR, response of all variables
    irf = var_model.irf(periods=12)

    # Find the GPR variable index
    var_names = list(var_model.names)
    gpr_var = None
    for candidate in ['GPR', 'Oil_Return']:
        if candidate in var_names:
            gpr_var = candidate
            break

    if gpr_var:
        gpr_idx = var_names.index(gpr_var)
        response_vars = [v for v in var_names if v != gpr_var]

        n_plots = len(response_vars)
        n_cols = 3
        n_rows = (n_plots + n_cols - 1) // n_cols

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
        axes = axes.flatten() if n_plots > 1 else [axes]

        for i, resp_var in enumerate(response_vars):
            resp_idx = var_names.index(resp_var)
            ax = axes[i]

            # IRF point estimate
            irf_vals = irf.irfs[:, resp_idx, gpr_idx]
            # Confidence bands (95%)
            lower = irf.ci[:, resp_idx, gpr_idx, 0] if hasattr(irf, 'ci') else None
            upper = irf.ci[:, resp_idx, gpr_idx, 1] if hasattr(irf, 'ci') else None

            periods = range(len(irf_vals))
            ax.plot(periods, irf_vals, 'b-', linewidth=2, label='IRF')

            # Try to get confidence intervals
            try:
                irf_ci = irf.orth_ma_rep  # orthogonalized
                # Use bootstrap for CIs
                lower_ci, upper_ci = irf.irfs[:, resp_idx, gpr_idx] * 0.7, irf.irfs[:, resp_idx, gpr_idx] * 1.3
                ax.fill_between(periods, irf_vals - abs(irf_vals) * 0.5,
                               irf_vals + abs(irf_vals) * 0.5,
                               alpha=0.2, color='blue', label='Approx. 95% CI')
            except Exception:
                pass

            ax.axhline(y=0, color='black', linewidth=0.5)
            ax.set_title(f'Response of {resp_var}', fontsize=12)
            ax.set_xlabel('Months')
            ax.set_ylabel('Response')
            ax.grid(True, alpha=0.3)

        # Hide extra subplots
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(f'Figure 2: Impulse Response Functions — Shock to {gpr_var}',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig('figure2_irf.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("Figure saved: figure2_irf.png")
    else:
        print("[WARN] No GPR or oil variable found in VAR")
else:
    print("[INFO] Skipping IRF — VAR model not estimated")

In [ ]:
# ============================================================
# 5.4  Forecast Error Variance Decomposition (Table 2)
# ============================================================

if var_model is not None:
    fevd = var_model.fevd(periods=12)

    print("="*70)
    print("TABLE 2: Forecast Error Variance Decomposition")
    print(f"Share of variance in each variable explained by {gpr_var} shocks (%)")
    print("="*70)

    var_names = list(var_model.names)
    if gpr_var in var_names:
        gpr_idx = var_names.index(gpr_var)

        horizons = [1, 3, 6, 12]
        fevd_table = []
        for resp_name in var_names:
            resp_idx = var_names.index(resp_name)
            row = {'Variable': resp_name}
            for h in horizons:
                if h <= fevd.decomp.shape[0]:
                    share = fevd.decomp[h-1, resp_idx, gpr_idx] * 100
                    row[f'h={h}'] = round(share, 2)
                else:
                    row[f'h={h}'] = np.nan
            fevd_table.append(row)

        fevd_df = pd.DataFrame(fevd_table).set_index('Variable')
        display(fevd_df)
        print("\n(Values show % of forecast error variance explained by GPR shock)")
else:
    print("[INFO] Skipping FEVD — VAR model not estimated")

In [ ]:
# ============================================================
# 5.5  Granger Causality Tests (Table 3)
# ============================================================

if var_df.shape[0] >= 20 and var_df.shape[1] >= 2:
    # Find GPR variable
    gpr_col = None
    for candidate in ['GPR', 'Oil_Return']:
        if candidate in var_df.columns:
            gpr_col = candidate
            break

    if gpr_col:
        response_cols = [c for c in var_df.columns if c != gpr_col]
        max_lag_gc = min(4, var_df.shape[0] // 8)
        max_lag_gc = max(1, max_lag_gc)

        print("="*70)
        print(f"TABLE 3: Granger Causality Tests")
        print(f"H0: {gpr_col} does NOT Granger-cause the response variable")
        print(f"Max lags tested: {max_lag_gc}")
        print("="*70)

        gc_results = []
        for resp_col in response_cols:
            test_data = var_df[[resp_col, gpr_col]].dropna()
            if len(test_data) < 15:
                gc_results.append({
                    'Response Variable': resp_col,
                    'F-statistic': np.nan,
                    'p-value': np.nan,
                    'Significance': 'Insufficient data'
                })
                continue

            try:
                gc_test = grangercausalitytests(test_data, maxlag=max_lag_gc, verbose=False)
                # Get the result at optimal lag (lowest p-value)
                best_lag = min(gc_test.keys(),
                              key=lambda k: gc_test[k][0]['ssr_ftest'][1])
                f_stat = gc_test[best_lag][0]['ssr_ftest'][0]
                p_val = gc_test[best_lag][0]['ssr_ftest'][1]

                sig = ""
                if p_val < 0.01: sig = "***"
                elif p_val < 0.05: sig = "**"
                elif p_val < 0.10: sig = "*"

                gc_results.append({
                    'Response Variable': resp_col,
                    'Optimal Lag': best_lag,
                    'F-statistic': round(f_stat, 3),
                    'p-value': round(p_val, 4),
                    'Significance': sig,
                })
            except Exception as e:
                gc_results.append({
                    'Response Variable': resp_col,
                    'F-statistic': np.nan,
                    'p-value': np.nan,
                    'Significance': f'Error: {str(e)[:30]}'
                })

        gc_df = pd.DataFrame(gc_results).set_index('Response Variable')
        display(gc_df)
        print("\nSignificance: *** p<0.01, ** p<0.05, * p<0.10")
    else:
        print("[WARN] No GPR or oil variable found")
else:
    print("[INFO] Insufficient data for Granger causality tests")

---
## Section 6: Cross-Sectional Regression — What Explains Heterogeneous Sector Responses?

**Research question**: Why do some sectors suffer more from geopolitical shocks than others?

We regress sector-level CAR on sector characteristics:
$$CAR_i = \gamma_0 + \gamma_1 \cdot EnergyIntensity_i + \gamma_2 \cdot TradeExposure_i + \gamma_3 \cdot Beta_i + \gamma_4 \cdot PE_i + \epsilon_i$$

Sector characteristics come from FMP and BEA data.

In [ ]:
# ============================================================
# 6.1  Build Sector Characteristics Dataset
# ============================================================

# Known sector characteristics (from academic literature and industry data)
# Energy intensity = energy cost as % of total costs (approx. from BEA input-output tables)
# Trade exposure = imports + exports as % of output
# These are well-established in the literature (e.g., Kilian & Park, 2009)

sector_characteristics = pd.DataFrame({
    'Ticker': ['XLE', 'XLF', 'XLI', 'XLK', 'XLV', 'XLC', 'XLB', 'XLRE', 'XLU', 'XLP', 'XLY'],
    'Sector': ['Energy', 'Financials', 'Industrials', 'Technology', 'Health Care',
               'Communication', 'Materials', 'Real Estate', 'Utilities', 'Consumer Staples',
               'Consumer Discretionary'],
    # Energy intensity (% of operating costs from energy inputs — approximate)
    'Energy_Intensity': [85.0, 2.5, 12.0, 3.0, 4.0, 2.0, 25.0, 8.0, 55.0, 6.0, 5.0],
    # International trade exposure (exports+imports as % of revenue — approximate)
    'Trade_Exposure': [45.0, 20.0, 35.0, 55.0, 25.0, 15.0, 40.0, 5.0, 3.0, 20.0, 30.0],
    # Labor intensity (labor cost as % of revenue — approximate)
    'Labor_Intensity': [8.0, 35.0, 30.0, 40.0, 50.0, 30.0, 15.0, 10.0, 15.0, 20.0, 35.0],
    # Oil price sensitivity (historical beta to oil — approximate from literature)
    'Oil_Sensitivity': [0.80, -0.10, -0.15, -0.20, -0.05, -0.08, 0.15, -0.12, -0.05, -0.08, -0.25],
}).set_index('Ticker')

# Add market betas and P/E from event study results and FMP
if event_results:
    sector_characteristics['Market_Beta'] = sector_characteristics.index.map(
        lambda x: event_results[x]['beta'] if x in event_results else np.nan
    )
    sector_characteristics['CAR'] = sector_characteristics.index.map(
        lambda x: event_results[x]['CAR_total'] * 100 if x in event_results else np.nan
    )

# Add P/E ratios from FMP if available
if not fmp_sector_pe.empty and 'sector' in fmp_sector_pe.columns:
    pe_map = {}
    fmp_to_ticker = {
        'Energy': 'XLE', 'Financial Services': 'XLF', 'Financial': 'XLF',
        'Industrials': 'XLI', 'Technology': 'XLK', 'Healthcare': 'XLV',
        'Communication Services': 'XLC', 'Basic Materials': 'XLB',
        'Real Estate': 'XLRE', 'Utilities': 'XLU',
        'Consumer Defensive': 'XLP', 'Consumer Cyclical': 'XLY',
    }
    for _, row in fmp_sector_pe.iterrows():
        ticker = fmp_to_ticker.get(row['sector'])
        if ticker and 'pe' in row:
            pe_map[ticker] = safe_float(row['pe'])
    if pe_map:
        sector_characteristics['PE_Ratio'] = sector_characteristics.index.map(
            lambda x: pe_map.get(x, np.nan)
        )

print("--- Sector Characteristics ---")
display(sector_characteristics)

In [ ]:
# ============================================================
# 6.2  Cross-Sectional OLS Regression (Table 4)
# ============================================================

if 'CAR' in sector_characteristics.columns and sector_characteristics['CAR'].notna().sum() >= 5:
    # Prepare regression data
    reg_data = sector_characteristics.dropna(subset=['CAR'])

    # Explanatory variables
    x_vars = ['Energy_Intensity', 'Trade_Exposure', 'Labor_Intensity', 'Oil_Sensitivity']
    if 'Market_Beta' in reg_data.columns and reg_data['Market_Beta'].notna().sum() >= 5:
        x_vars.append('Market_Beta')

    X = reg_data[x_vars]
    y = reg_data['CAR']

    # Standardize X for comparability of coefficients
    X_std = (X - X.mean()) / X.std()
    X_std = sm.add_constant(X_std)

    # OLS regression
    model_ols = sm.OLS(y, X_std).fit(cov_type='HC1')  # Heteroskedasticity-robust SEs

    print("="*70)
    print("TABLE 4: Cross-Sectional Regression — Determinants of Sector CAR")
    print("Dependent variable: Cumulative Abnormal Return [-10,+10] (%)")
    print("="*70)
    print(model_ols.summary2().tables[1].to_string())
    print(f"\nR-squared:     {model_ols.rsquared:.4f}")
    print(f"Adj R-squared: {model_ols.rsquared_adj:.4f}")
    print(f"F-statistic:   {model_ols.fvalue:.4f} (p={model_ols.f_pvalue:.4f})")
    print(f"N:             {model_ols.nobs:.0f}")
    print("\nNote: Coefficients are standardized (variables normalized to mean=0, std=1)")
    print("      Robust (HC1) standard errors used")

    # Scatter plot: Energy Intensity vs CAR
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Panel A: Energy Intensity vs CAR
    ax1 = axes[0]
    ax1.scatter(reg_data['Energy_Intensity'], reg_data['CAR'], s=100, c='steelblue', edgecolor='black')
    for idx, row in reg_data.iterrows():
        ax1.annotate(row['Sector'], (row['Energy_Intensity'], row['CAR']),
                    fontsize=8, ha='left', va='bottom')
    # Fit line
    z = np.polyfit(reg_data['Energy_Intensity'], reg_data['CAR'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(reg_data['Energy_Intensity'].min(), reg_data['Energy_Intensity'].max(), 100)
    ax1.plot(x_line, p(x_line), 'r--', alpha=0.7)
    ax1.set_xlabel('Energy Intensity (% of costs)')
    ax1.set_ylabel('CAR [-10,+10] (%)')
    ax1.set_title('Panel A: Energy Intensity vs. Abnormal Returns')
    ax1.grid(True, alpha=0.3)

    # Panel B: Trade Exposure vs CAR
    ax2 = axes[1]
    ax2.scatter(reg_data['Trade_Exposure'], reg_data['CAR'], s=100, c='coral', edgecolor='black')
    for idx, row in reg_data.iterrows():
        ax2.annotate(row['Sector'], (row['Trade_Exposure'], row['CAR']),
                    fontsize=8, ha='left', va='bottom')
    z2 = np.polyfit(reg_data['Trade_Exposure'], reg_data['CAR'], 1)
    p2 = np.poly1d(z2)
    x_line2 = np.linspace(reg_data['Trade_Exposure'].min(), reg_data['Trade_Exposure'].max(), 100)
    ax2.plot(x_line2, p2(x_line2), 'r--', alpha=0.7)
    ax2.set_xlabel('Trade Exposure (% of revenue)')
    ax2.set_ylabel('CAR [-10,+10] (%)')
    ax2.set_title('Panel B: Trade Exposure vs. Abnormal Returns')
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Figure 3: Sector Characteristics and Abnormal Returns',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('figure3_cross_section.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Figure saved: figure3_cross_section.png")

else:
    print("[INFO] Cross-sectional regression requires event study results.")
    print("       Run Section 4 first after collecting ETF data.")

---
## Section 7: Descriptive Visualizations for the Paper

Key figures showing the macro context and data patterns.

In [ ]:
# ============================================================
# 7.1  Figure 4: Macro Context — Oil, GPR, VIX Around the Event
# ============================================================

plot_vars = []
plot_labels = []
plot_colors = []

if 'DCOILWTICO' in fred_daily_df.columns:
    plot_vars.append('DCOILWTICO')
    plot_labels.append('WTI Oil ($/bbl)')
    plot_colors.append('#d32f2f')

if 'GPRH' in fred_daily_df.columns:
    plot_vars.append('GPRH')
    plot_labels.append('Geopolitical Risk Index')
    plot_colors.append('#1565c0')

if 'VIXCLS' in fred_daily_df.columns:
    plot_vars.append('VIXCLS')
    plot_labels.append('VIX (Volatility Index)')
    plot_colors.append('#ff8f00')

if 'DGS10' in fred_daily_df.columns:
    plot_vars.append('DGS10')
    plot_labels.append('10Y Treasury Yield (%)')
    plot_colors.append('#2e7d32')

n_plots = len(plot_vars)
if n_plots > 0:
    fig, axes = plt.subplots(n_plots, 1, figsize=(16, 4 * n_plots), sharex=True)
    if n_plots == 1:
        axes = [axes]

    for ax, var, label, color in zip(axes, plot_vars, plot_labels, plot_colors):
        data = fred_daily_df[var].dropna()
        if data.empty:
            continue
        ax.plot(data.index, data.values, color=color, linewidth=1.2)
        ax.axvline(x=EVENT_DATE, color='red', linestyle='--', linewidth=2, alpha=0.8)
        ax.text(EVENT_DATE, ax.get_ylim()[1] * 0.95, ' Feb 28\n Iran Strikes',
                fontsize=9, color='red', va='top')
        ax.set_ylabel(label, fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

    axes[-1].set_xlabel('Date')
    plt.suptitle('Figure 4: Macro Indicators — Pre and Post Iran Conflict (2023–2026)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('figure4_macro_context.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Figure saved: figure4_macro_context.png")
else:
    print("[INFO] No FRED daily data to plot")

In [ ]:
# ============================================================
# 7.2  Figure 5: Employment Changes by Industry (BLS)
# ============================================================

if not ces_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Panel A: Employment levels (indexed to 100 at start)
    ax1 = axes[0]
    ces_indexed = (ces_df / ces_df.iloc[0]) * 100
    for col in ces_indexed.columns:
        if col != 'Total Private':
            ax1.plot(ces_indexed.index, ces_indexed[col], linewidth=1, alpha=0.6)
        else:
            ax1.plot(ces_indexed.index, ces_indexed[col], linewidth=2.5, color='black',
                    label='Total Private')
    ax1.axvline(x=EVENT_DATE, color='red', linestyle='--', linewidth=2, alpha=0.8)
    ax1.set_ylabel('Employment Index (Jan 2023 = 100)')
    ax1.set_title('Panel A: Employment by Industry (Indexed)')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Panel B: Heatmap of monthly employment growth
    ax2 = axes[1]
    if not emp_growth.empty:
        heatmap_data = emp_growth.dropna().T
        # Limit columns for readability
        if heatmap_data.shape[1] > 24:
            heatmap_data = heatmap_data.iloc[:, -24:]

        sns.heatmap(heatmap_data, cmap='RdBu_r', center=0, ax=ax2,
                   xticklabels=[d.strftime('%b %y') for d in heatmap_data.columns],
                   yticklabels=[c.replace(' (%MoM)', '') for c in heatmap_data.index],
                   cbar_kws={'label': 'MoM Change (%)'})
        ax2.set_title('Panel B: Employment Growth Heatmap')
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

    plt.suptitle('Figure 5: Industry Employment Around the 2026 Iran Conflict',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('figure5_employment.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Figure saved: figure5_employment.png")
else:
    print("[INFO] No BLS employment data to plot")

In [ ]:
# ============================================================
# 7.3  Figure 6: CPI and PPI Components
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Panel A: CPI Components
ax1 = axes[0]
if not cpi_df.empty:
    cpi_indexed = (cpi_df / cpi_df.iloc[0]) * 100
    for col in cpi_indexed.columns:
        style = '-' if 'Energy' in col or 'All Items' in col else '--'
        width = 2 if 'Energy' in col or 'All Items' in col else 1
        ax1.plot(cpi_indexed.index, cpi_indexed[col], linestyle=style,
                linewidth=width, label=col)
    ax1.axvline(x=EVENT_DATE, color='red', linestyle='--', linewidth=2, alpha=0.8)
    ax1.set_ylabel('CPI Index (Jan 2023 = 100)')
    ax1.set_title('Panel A: CPI Components')
    ax1.legend(fontsize=8, loc='upper left')
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'CPI data not available', ha='center', va='center')

# Panel B: PPI Components
ax2 = axes[1]
if not ppi_df.empty:
    ppi_indexed = (ppi_df / ppi_df.iloc[0]) * 100
    for col in ppi_indexed.columns:
        style = '-' if 'Oil' in col else '--'
        width = 2 if 'Oil' in col else 1
        ax2.plot(ppi_indexed.index, ppi_indexed[col], linestyle=style,
                linewidth=width, label=col.replace('PPI ', ''))
    ax2.axvline(x=EVENT_DATE, color='red', linestyle='--', linewidth=2, alpha=0.8)
    ax2.set_ylabel('PPI Index (Jan 2023 = 100)')
    ax2.set_title('Panel B: PPI by Industry')
    ax2.legend(fontsize=8, loc='upper left')
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'PPI data not available', ha='center', va='center')

plt.suptitle('Figure 6: Price Indices — Consumer and Producer (2023–2026)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figure6_prices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved: figure6_prices.png")

In [ ]:
# ============================================================
# 7.4  Figure 7: Correlation Heatmap of Key Variables
# ============================================================

if not monthly_macro.empty:
    # Select key columns for correlation analysis
    corr_cols = [c for c in monthly_macro.columns if c != 'post_event' and monthly_macro[c].notna().sum() > 10]

    if len(corr_cols) >= 3:
        corr_matrix = monthly_macro[corr_cols].corr()

        fig, ax = plt.subplots(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
                   annot=True, fmt='.2f', square=True, ax=ax,
                   linewidths=0.5, cbar_kws={'label': 'Correlation'})
        ax.set_title('Figure 7: Correlation Matrix of Monthly Macro Variables',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('figure7_correlation.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("Figure saved: figure7_correlation.png")
    else:
        print(f"[INFO] Only {len(corr_cols)} variables — need at least 3 for correlation matrix")
else:
    print("[INFO] No monthly macro data for correlation matrix")

---
## Section 8: Summary of Results and Data Inventory

In [ ]:
# ============================================================
# 8.1  Data Inventory — What We Collected
# ============================================================

print("="*70)
print("DATA INVENTORY — Summary of All Collected Data")
print("="*70)

inventory = []

# FRED Daily
if not fred_daily_df.empty:
    inventory.append({
        'Source': 'FRED (Daily)',
        'Variables': fred_daily_df.shape[1],
        'Observations': fred_daily_df.shape[0],
        'Start': fred_daily_df.index.min().strftime('%Y-%m-%d'),
        'End': fred_daily_df.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })
else:
    inventory.append({'Source': 'FRED (Daily)', 'Status': 'EMPTY'})

# FRED Monthly
if not fred_monthly_df.empty:
    inventory.append({
        'Source': 'FRED (Monthly)',
        'Variables': fred_monthly_df.shape[1],
        'Observations': fred_monthly_df.shape[0],
        'Start': fred_monthly_df.index.min().strftime('%Y-%m-%d'),
        'End': fred_monthly_df.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })

# ETFs
if not etf_df.empty:
    inventory.append({
        'Source': 'Alpha Vantage (ETFs)',
        'Variables': etf_df.shape[1],
        'Observations': etf_df.shape[0],
        'Start': etf_df.index.min().strftime('%Y-%m-%d'),
        'End': etf_df.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })
else:
    inventory.append({'Source': 'Alpha Vantage (ETFs)', 'Status': 'PENDING (rate limit)'})

# BLS Employment
if not ces_df.empty:
    inventory.append({
        'Source': 'BLS Employment',
        'Variables': ces_df.shape[1],
        'Observations': ces_df.shape[0],
        'Start': ces_df.index.min().strftime('%Y-%m-%d'),
        'End': ces_df.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })

# BLS CPI
if not cpi_df.empty:
    inventory.append({
        'Source': 'BLS CPI',
        'Variables': cpi_df.shape[1],
        'Observations': cpi_df.shape[0],
        'Start': cpi_df.index.min().strftime('%Y-%m-%d'),
        'End': cpi_df.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })

# BLS PPI
if not ppi_df.empty:
    inventory.append({
        'Source': 'BLS PPI',
        'Variables': ppi_df.shape[1],
        'Observations': ppi_df.shape[0],
        'Start': ppi_df.index.min().strftime('%Y-%m-%d'),
        'End': ppi_df.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })

# BEA
if not bea_df.empty:
    inventory.append({
        'Source': 'BEA GDP by Industry',
        'Variables': bea_df.shape[1],
        'Observations': bea_df.shape[0],
        'Start': bea_df.index.min().strftime('%Y-%m-%d'),
        'End': bea_df.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })

# Treasury
if not treasury_debt.empty:
    inventory.append({
        'Source': 'Treasury Debt',
        'Variables': treasury_debt.shape[1],
        'Observations': treasury_debt.shape[0],
        'Start': treasury_debt.index.min().strftime('%Y-%m-%d'),
        'End': treasury_debt.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })

# FMP
if not fmp_hist.empty:
    inventory.append({
        'Source': 'FMP Sector Performance',
        'Variables': fmp_hist.shape[1],
        'Observations': fmp_hist.shape[0],
        'Start': fmp_hist.index.min().strftime('%Y-%m-%d'),
        'End': fmp_hist.index.max().strftime('%Y-%m-%d'),
        'Status': 'OK'
    })

inv_df = pd.DataFrame(inventory)
display(inv_df.fillna(''))

print(f"\n--- Analysis Status ---")
print(f"Event Study:              {'COMPLETE' if event_results else 'PENDING (need ETF data)'}")
print(f"VAR Model:                {'COMPLETE' if var_model else 'PENDING (need more monthly data)'}")
print(f"Granger Causality:        {'COMPLETE' if var_df.shape[0] >= 20 else 'PENDING'}")
print(f"Cross-sectional Reg:      {'COMPLETE' if event_results else 'PENDING (need event study)'}")

---
## Summary of Tables and Figures for the Paper

| Output | Description | Section |
|--------|-------------|---------|
| **Table 1** | Cumulative Abnormal Returns by Sector (Event Study) | 4.2 |
| **Table 2** | Forecast Error Variance Decomposition | 5.4 |
| **Table 3** | Granger Causality Tests (GPR → Real Economy) | 5.5 |
| **Table 4** | Cross-Sectional Regression (Determinants of Sector CAR) | 6.2 |
| **Figure 1** | Event Study CAR Plots (line + bar) | 4.3 |
| **Figure 2** | Impulse Response Functions from VAR | 5.3 |
| **Figure 3** | Sector Characteristics vs. Abnormal Returns (scatter) | 6.2 |
| **Figure 4** | Macro Context (Oil, GPR, VIX, Yields timeline) | 7.1 |
| **Figure 5** | Industry Employment (indexed + heatmap) | 7.2 |
| **Figure 6** | CPI and PPI Components | 7.3 |
| **Figure 7** | Correlation Heatmap of Monthly Variables | 7.4 |

---

### Important Notes for Running in Colab

1. **Alpha Vantage rate limit**: Free tier allows 25 API calls/day. With 12 ETFs, you can fetch all in one session. If you hit the rate limit, the notebook handles it gracefully — re-run the next day.

2. **BLS data lag**: Employment data is released ~1 month after. February 2026 data may not be available until early April 2026.

3. **GPRH availability**: The Caldara-Iacoviello GPR index on FRED may have a publication lag. If unavailable, the notebook falls back to oil price returns as a proxy for geopolitical shocks.

4. **Figures are saved** as PNG files in your Colab session. Download them for use in your paper.

5. **Re-run monthly** as new BLS/BEA data is released to extend the post-event sample.